# Explore Butler regristry for LSSTCam

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-19

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import lsst.afw.display as afwDisplay
import numpy as np
import pandas as pd
from astropy.time import Time

In [ ]:
from lsst.daf.butler import Butler

In [ ]:
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.labelsize"] = "x-large"
plt.rcParams["axes.titlesize"] = "x-large"
plt.rcParams["xtick.labelsize"] = "x-large"
plt.rcParams["ytick.labelsize"] = "x-large"


In [ ]:
!eups list lsst_distrib

In [ ]:
repo = "main"
instrument = "LSSTCam"
date_start = 20250415
date_selection = 20250416
where_clause = "instrument = '" + f"{instrument}" + "'"
where_clause_date = where_clause + f"and day_obs >= {date_start}"
skymapName = "lsst_cells_v2"
COLLECTION_NAME = 'LSSTCam/raw/all' 

In [ ]:
os.getenv("LSST_RESOURCES_WEBDAV_CONFIG")

In [ ]:
os.getenv("DAF_BUTLER_REPOSITORY_INDEX")

## Butler init

In [ ]:
#!eups list -s

In [ ]:
butler = Butler(repo)
registry = butler.registry

In [ ]:
butler.registry.queryDatasetTypes()

In [ ]:
all_collections = registry.queryCollections()

print(f"Nombre total de collections trouvées : {len(all_collections)}")

# Afficher les 20 premières pour un aperçu
print("\nLes 20 premières collections :")
for i, collection_name in enumerate(all_collections):
    print(collection_name)
    if i >= 19:
        break

In [ ]:
# Essayez la même méthode, mais sur l'objet 'registry'
try:
    exposure_records = registry.queryDimensionRecords('exposure')
    
    # Le reste de votre code
    list_of_records = list(exposure_records)
    print(f"Nombre total d'expositions trouvées : {len(list_of_records)}")

except AttributeError as e:
    print(f"Erreur de l'API Gen 3 : {e}")
    print("Veuillez vérifier l'initialisation de votre Butler, notamment l'URI et les collections.")

In [ ]:
df_exposure = pd.DataFrame(
    {
        "id": pd.Series(dtype="int"),
        "obs_id": pd.Series(dtype="int"),
        "day_obs": pd.Series(dtype="int"),
        "seq_num": pd.Series(dtype="int"),
        "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c’est un datetime
        "time_end": pd.Series(dtype="str"),  # idem
        "type": pd.Series(dtype="str"),
        "target": pd.Series(dtype="str"),
        "filter": pd.Series(dtype="str"),
        "zenith_angle": pd.Series(dtype="float"),
        "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
        "ra": pd.Series(dtype="float"),
        "dec": pd.Series(dtype="float"),
        "skyangle": pd.Series(dtype="float"),
        "azimuth": pd.Series(dtype="float"),
        "zenith": pd.Series(dtype="float"),
        "science_program": pd.Series(dtype="str"),
        "jd": pd.Series(dtype="float"),
        "mjd": pd.Series(dtype="float"),
    }
)

In [ ]:
# save the data array in rows before saving in pandas dataframe
rows = []
for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
    try:
        jd_start = info.timespan.begin.value
        jd_end = info.timespan.end.value
        the_Time_start = Time(jd_start, format="jd", scale="utc")
        the_Time_end = Time(jd_end, format="jd", scale="utc")
        mjd_start = the_Time_start.mjd
        mjd_end = the_Time_end.mjd
        isot_start = the_Time_start.isot
        isot_end = the_Time_end.isot

        if count == 0:
            print("===== Time Conversion Debug Info =====")
            print(f"JD start      : {jd_start} (type: {type(jd_start)})")
            print(f"JD end        : {jd_end} (type: {type(jd_end)})")
            print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
            print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
            print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
            print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
            print("=======================================")

        # put row in a dictionnary before stacking
        row = {
            "id": info.id,
            "obs_id": info.obs_id,
            "day_obs": info.day_obs,
            "seq_num": info.seq_num,
            "time_start": isot_start,
            "time_end": isot_end,
            "type": info.observation_type,
            "target": info.target_name,
            "filter": info.physical_filter,
            "zenith_angle": info.zenith_angle,
            "expos": info.exposure_time,  # Exemple : adapter selon ton objet
            "ra": info.tracking_ra,
            "dec": info.tracking_dec,
            "skyangle": info.sky_angle,
            "azimuth": info.azimuth,
            "zenith": info.zenith_angle,
            "science_program": info.science_program,
            "jd": float(jd_start),
            "mjd": float(mjd_start),
        }
        rows.append(row)

    except ValueError as e:
        print(f"Erreur de valeur : {e}")
    except FileNotFoundError as e:
        print(f"Fichier introuvable : {e}")
    except Exception as e:
        print(f"Erreur inattendue : {type(e).__name__} - {e}")
        print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
        traceback.print_exc()  # affiche la stack trace complète


In [ ]:
# Création finale du DataFrame
df_exposure = pd.DataFrame(rows)

In [ ]:
df_exposure

In [ ]:
df_exposure["target"].unique()

In [ ]:
df_science = df_exposure[df_exposure.type == "science"]
df_science.reset_index(drop=True, inplace=True)

In [ ]:
df_science = df_science.copy()
df_science["band"] = df_science["filter"].apply(lambda x : x.split("_")[0])

In [ ]:
df_science

## Plot

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_pernight_visits_per_band(
    df,
    bands=("u", "g", "r", "i", "z", "y"),
    figsize=(15, 7),
    date_rotation=45,
    show_legend=True,
):
    """
    Plot number of visits per night, per filter band, night by night.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing at least 'mjd', 'day_obs' and 'band'.
    bands : tuple
        Filter bands to display.
    figsize : tuple
        Figure size.
    date_rotation : float
        Rotation angle of the upper date axis labels.
    show_legend : bool
        Whether to display the legend.

    Returns
    -------
    fig, ax
        Matplotlib figure and primary axis.
    """

    # Work on a copy
    data = df.copy()

    # Keep valid science observations
    data = data.dropna(subset=["mjd", "day_obs", "band"])

    # Convert dates
    data["day_obs"] = pd.to_datetime(
        data["day_obs"].astype(str),
        format="%Y%m%d"
    )

    # Sort observations chronologically
    data = data.sort_values("mjd").reset_index(drop=True)

    # Compute cumulative number of visits per band and per night
    data["n_day"] = (
        data.groupby(["day_obs", "band"])
        .cumcount() + 1
    )

    # Define colors for LSST bands
    band_colors = {
        "u": "tab:purple",
        "g": "tab:blue",
        "r": "tab:green",
        "i": "tab:orange",
        "z": "tab:red",
        "y": "tab:brown",
    }

    fig, ax = plt.subplots(figsize=figsize)

    # Plot each band, night by night
    for band in bands:

        band_data = data[data["band"] == band]

        for night, night_data in band_data.groupby("day_obs"):

            ax.plot(
                night_data["mjd"],
                night_data["n_day"],
                color=band_colors.get(band, "gray"),
                linewidth=1.5,
                marker=".",
                markersize=3,
                label=band if night == band_data["day_obs"].iloc[0]
                else None,
            )

    # Primary x-axis: MJD
    ax.set_xlabel("Modified Julian Date (MJD)", fontsize=12)

    # Y-axis
    ax.set_ylabel("Cumulative number of visits per night", fontsize=12)

    ax.set_ylim(bottom=0)

    # Grid
    ax.grid(True, alpha=0.3, linestyle="--")

    # Secondary x-axis: calendar dates
    ax_top = ax.twiny()

    ax_top.set_xlim(ax.get_xlim())

    # Use the same MJD-to-date conversion for tick positions
    mjd_ticks = ax.get_xticks()
    date_ticks = pd.to_datetime(
        mjd_ticks + 2400000.5,
        unit="D",
        origin="julian"
    )

    ax_top.set_xticks(mjd_ticks)
    ax_top.set_xticklabels(
        date_ticks.strftime("%Y/%m/%d"),
        rotation=date_rotation,
        ha="left"
    )

    ax_top.set_xlabel("Observation date", fontsize=12, labelpad=10)

    # Legend
    if show_legend:
        handles, labels = ax.get_legend_handles_labels()

        # Remove duplicate labels
        unique = dict(zip(labels, handles))

        ax.legend(
            unique.values(),
            unique.keys(),
            title="Filter band",
            loc="upper left",
            frameon=True,
        )

    fig.tight_layout()

    return fig, ax


In [ ]:
fig, ax = plot_pernight_visits_per_band(df_science)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator


def plot_cumulative_visits_per_band(
    df,
    bands=("u", "g", "r", "i", "z", "y"),
    figsize=(15, 7),
    date_rotation=45,
):
    """
    Plot global cumulative visits per filter band,
    with one bin per observing night.

    Bottom x-axis: numerical MJD values.
    Top x-axis: calendar dates (YYYY/MM/DD).
    """

    # --------------------------------------------------
    # 1. Prepare data
    # --------------------------------------------------

    data = df.dropna(
        subset=["mjd", "day_obs", "band"]
    ).copy()

    data["day_obs"] = pd.to_datetime(
        data["day_obs"].astype(str),
        format="%Y%m%d"
    )

    # --------------------------------------------------
    # 2. Count visits per night and per band
    # --------------------------------------------------

    daily_counts = (
        data.groupby(["day_obs", "band"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=bands, fill_value=0)
        .sort_index()
    )

    all_nights = pd.date_range(
        daily_counts.index.min(),
        daily_counts.index.max(),
        freq="D"
    )

    daily_counts = daily_counts.reindex(
        all_nights,
        fill_value=0
    )

    # --------------------------------------------------
    # 3. Global cumulative sum
    # --------------------------------------------------

    cumulative = daily_counts.cumsum()

    # Convert observing dates to MJD
    mjd_nights = (
        cumulative.index - pd.Timestamp("1858-11-17")
    ).total_seconds() / 86400.0

    # --------------------------------------------------
    # 4. Colors
    # --------------------------------------------------

    band_colors = {
        "u": "tab:purple",
        "g": "tab:blue",
        "r": "tab:green",
        "i": "tab:orange",
        "z": "tab:red",
        "y": "tab:brown",
    }

    # --------------------------------------------------
    # 5. Plot
    # --------------------------------------------------

    fig, ax = plt.subplots(figsize=figsize)

    for band in bands:
        ax.step(
            mjd_nights,
            cumulative[band].values,
            where="post",
            color=band_colors.get(band, "gray"),
            linewidth=2,
            label=band,
        )

    # --------------------------------------------------
    # 6. Bottom axis: numerical MJD
    # --------------------------------------------------

    ax.set_xlabel("Modified Julian Date (MJD)", fontsize=12)
    ax.set_ylabel("Cumulative number of visits", fontsize=12)

    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3, linestyle="--")

    # Force numerical MJD ticks
    ax.xaxis.set_major_locator(MaxNLocator(nbins=10))
    ax.ticklabel_format(axis="x", style="plain", useOffset=False)

    # --------------------------------------------------
    # 7. Top axis: calendar dates
    # --------------------------------------------------

    ax_top = ax.twiny()

    # Synchronize limits
    ax_top.set_xlim(ax.get_xlim())

    # Use exactly the same tick positions
    mjd_ticks = ax.get_xticks()
    ax_top.set_xticks(mjd_ticks)

    # Convert MJD to calendar dates
    date_ticks = pd.to_datetime(
        mjd_ticks,
        unit="D",
        origin=pd.Timestamp("1858-11-17")
    )

    ax_top.set_xticklabels(
        date_ticks.strftime("%Y/%m/%d"),
        rotation=date_rotation,
        ha="left"
    )

    ax_top.set_xlabel("Observation date", fontsize=12, labelpad=10)

    # --------------------------------------------------
    # 8. Legend and layout
    # --------------------------------------------------

    ax.legend(
        title="Filter band",
        loc="upper left",
        frameon=True
    )

    fig.tight_layout()

    return fig, ax


In [ ]:
fig, ax = plot_cumulative_visits_per_band(df_science)
plt.show()